# 因果语言训练示例

## Step1 导入相关包

In [ ]:
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

## Step2 加载数据集

In [ ]:
dataset = load_from_disk("./wiki_cn_filtered/")
dataset

In [ ]:
dataset[0]

## Step3 数据集处理

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Langboat/bloom-389m-zh")

def process_func(examples):
    contents = [e+tokenizer.eos_token for e in examples["completion"]]
    return tokenizer(contents, max_length=384, trunctation=True)

In [ ]:
tokenized_dataset = dataset.map(process_func,batched=True, remove_columns=dataset.column_names)
tokenized_dataset

In [ ]:
from torch.utils.data import DataLoader

dataloder = DataLoader(tokenized_dataset, batch_size=2, collate_fn=DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False))

In [ ]:
next(iter(dataloder))

In [ ]:
dir(tokenizer)

In [ ]:
tokenizer.all_special_tokens, tokenizer.all_special_ids

## Step4 创建模型

In [ ]:
model = AutoModelForCausalLM.from_pretrained("Langboat/bloom-389m-zh")

## Step5 配置训练参数

In [ ]:
args = TrainingArguments(
    output_dir="./causal_lm",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    logging_steps=10,
    num_train_epochs=1,
)

## Stept6 创建Trainer

In [ ]:
trainer = Trainer(
    args=args,
    model=model,
    train_dataset=tokenized_dataset.select(range(100)),
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)


## Step7 模型训练

In [ ]:
trainer.train()

## Step8 模型推理

In [ ]:
from transformers import pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer,device=0)

In [ ]:
pipe("西安交通大学是一所",max_length=128,do_sample=True)